---

### The user story.  

As a traveller I want to upload beautiful photos I have taken on my travels, so I can visually see all the places I visited and when.

### The application architecture.  

![architecture](images/architecture.png)

---

### How to use this notebook

This is the companion to **"Anatomy of a Modern App"** — not a deployment guide (that's
[runbook.ipynb](runbook.ipynb) in this same folder). It assumes the stage-4 stack from that
runbook is **already up and synced** via Argo CD before you go on stage. Run the cell below once
to confirm; if anything is unreachable, go finish `runbook.ipynb` first.

The thread through everything below is one user action — uploading a photo — followed as it
ripples through five independent services, entirely asynchronously. Everything is driven from
[http://web.mytravels.local:8080](http://web.mytravels.local:8080) and cross-checked from here.

In [ ]:
%%bash
echo "=== Argo CD sync state (the source of truth for \"is the stack actually up\") ==="
kubectl get application mytravels -n argocd
echo ""
echo "=== Every host this talk will touch ==="
for host in web api mcp rabbitmq minio grafana prometheus solr flagsmith argocd; do
  URL="http://${host}.mytravels.local:8080"
  CODE=$(curl -s -o /dev/null -w "%{http_code}" --max-time 5 "$URL")
  printf "  %-10s %-45s %s\n" "$host" "$URL" "$CODE"
done

if ! curl -s -o /dev/null -m 5 http://api.mytravels.local:8080; then
  echo ""
  echo "api is unreachable — diagnosing inline:"
  kubectl get pods -n mytravels-default -o wide
  kubectl get application mytravels -n argocd \
    -o jsonpath='{.status.sync.status}: {.status.health.status}{"\n"}'
fi

---

## The Ecosystem at a Glance

One user story — *"as a traveller, I want to upload a photo and see it on a map"* — and it takes
a small mesh of independently-deployable services to deliver it:

| Service | Role | URL |
|---|---|---|
| `web` | React SPA — map, upload, traceability UI | [http://web.mytravels.local:8080](http://web.mytravels.local:8080) |
| `api` | REST API — the only synchronous door in | [http://api.mytravels.local:8080/swagger](http://api.mytravels.local:8080/swagger) |
| `mcp` | Same domain, exposed as MCP tools for LLM clients instead of REST | [http://mcp.mytravels.local:8080/health](http://mcp.mytravels.local:8080/health) |
| `messaging` | Background worker — every async step below runs here | (no UI — see RabbitMQ/traceability) |
| PostgreSQL | System of record for POIs, tags, audit logs | via `api` only |
| RabbitMQ | Message bus — four work exchanges + their `-failed` twins | [http://rabbitmq.mytravels.local:8080](http://rabbitmq.mytravels.local:8080) |
| MinIO | Object storage for original + resized images | [http://minio.mytravels.local:8080](http://minio.mytravels.local:8080) |
| SOLR | Search index — also what the map's list endpoint reads | [http://solr.mytravels.local:8080/solr/#/mytravels-pois/core-overview](http://solr.mytravels.local:8080/solr/#/mytravels-pois/core-overview) |
| Flagsmith | Feature flags, evaluated by both `.NET` and `web` | [http://flagsmith.mytravels.local:8080](http://flagsmith.mytravels.local:8080) |
| Prometheus / Grafana | Metrics + dashboards | [http://grafana.mytravels.local:8080](http://grafana.mytravels.local:8080) |
| Argo CD | GitOps control plane holding all of the above to `manifests/` | [http://argocd.mytravels.local:8080](http://argocd.mytravels.local:8080) |

Credentials everywhere in this tutorial are the same on purpose: `user123` / `password123` (read
from `.env` in the cells below rather than typed — same habit as `runbook.ipynb`).

---

## 1 — Upload a Photo, Watch It Ripple

Open [http://web.mytravels.local:8080](http://web.mytravels.local:8080) and upload a geotagged
photo (or, if it has no GPS EXIF, use the "search for the place instead" flow the upload menu
offers — `POST /api/pointofinterest/image/coordinates`). Come back here once it's done.

**What just happened synchronously**, inside one HTTP request: the original bytes landed in
MinIO's `uploaded-images` bucket, the photo's GPS EXIF was read, and one `PointOfInterests` row
was inserted with a freshly minted `CorrelationId`. That's it — the request returns before any
address, thumbnail, description or search entry exists.

**What happens next, asynchronously:** three messages are published in the same request, all
carrying that one `CorrelationId`:

| Exchange | Consumer | Fills in |
|---|---|---|
| `append-formatted-address` | `AppendFormattedAddress` | `FormattedAddress` (§2 below) |
| `resize-image` | `ResizeImage` | a thumbnail, then chains `append-image-tags` (§3–4) |
| `index-solr` | `IndexSolr` | makes the POI findable *right now*, address/description empty |

Five independent services, three separate consumers, zero coordination between them beyond a
shared id. Run the next cell to see the record before any of that has landed.

In [ ]:
%%bash
echo "=== Newest point of interest (fetched right after upload) ==="
curl -s "http://api.mytravels.local:8080/api/pointofinterest?rows=200" | python3 -c "
import json, sys
pois = json.load(sys.stdin)
if not pois:
    print('No points of interest yet — upload one at http://web.mytravels.local:8080 first.')
    sys.exit(0)
p = pois[-1]
print('  id                :', p['id'])
print('  pointOfInterestKey:', p['pointOfInterestKey'])
print('  correlationId     :', p.get('correlationId'))
print('  formattedAddress  :', repr(p.get('formattedAddress')), '(pending geocoding)' if not p.get('formattedAddress') else '')
print('  description       :', repr(p.get('description')), '(pending Anthropic call)' if not p.get('description') else '')
print('  tags              :', p.get('tags'))
print()
print('CORRELATION_ID=' + str(p.get('correlationId')))
" | tee /tmp/inside_scoop_poi.txt

CORR=$(grep '^CORRELATION_ID=' /tmp/inside_scoop_poi.txt | cut -d= -f2)
if [ -n "$CORR" ] && [ "$CORR" != "None" ]; then
  echo "$CORR" > /tmp/inside_scoop_correlation_id.txt
  echo ""
  echo "Saved for later cells: $CORR"
fi

---

## 2 — Address Lookup

`AppendFormattedAddress` calls a geocoding provider — **Google Maps** if `GoogleApiKey` is a real
key, or a silent, automatic fallback to **OpenStreetMap Nominatim** if it's unset or still the
placeholder. That fallback is a config difference, not a code path anyone has to choose — the app
doesn't know or care which provider answered.

Both providers get the same Polly policy: 2 retries with exponential backoff (~2s, ~4s), no
circuit breaker, no overall timeout. The consumer is idempotent — it skips resolution entirely if
`FormattedAddress` is already non-empty — so an at-least-once redelivery is harmless in the steady
state. A 30-minute sweeper additionally re-scans rows with an empty (not `NULL`) address from the
last 2 days, catching stragglers that missed their first pass.

Watch the same POI pick up its address:

In [ ]:
%%bash
API="http://api.mytravels.local:8080"
ID=$(curl -s "$API/api/pointofinterest?rows=200" | python3 -c "
import json, sys
pois = json.load(sys.stdin)
print(pois[-1]['id'] if pois else '')
")
if [ -z "$ID" ]; then
  echo "No points of interest yet — run the upload cell above first."
  exit 0
fi

echo "=== Polling id=$ID for formattedAddress (up to 60s) ==="
for i in $(seq 1 12); do
  ADDR=$(curl -s "$API/api/pointofinterest?rows=200" | python3 -c "
import json, sys
pois = json.load(sys.stdin)
poi = next((p for p in pois if p['id'] == $ID), None)
print(poi.get('formattedAddress') or '' if poi else '')
")
  if [ -n "$ADDR" ]; then
    echo "  resolved: $ADDR"
    exit 0
  fi
  echo "  attempt $i: still empty..."
  sleep 5
done

echo "No address after 60s — diagnosing inline:"
kubectl get pods -n mytravels-default -l app=messaging -o wide
kubectl logs -n mytravels-default -l app=messaging --tail=60 | grep -i -e address -e geocod || \
  kubectl logs -n mytravels-default -l app=messaging --tail=60

---

## 3 — Where Do Uploaded Images Go, and Why Resize?

Two MinIO buckets, both lazily auto-created on first write:

- **`uploaded-images`** — the original, full-resolution file, straight off the upload request.
- **`resized-images`** — a thumbnail at 10% of the original's dimensions, produced by the
  `ResizeImage` consumer.

Why bother resizing at all? The map renders potentially dozens of markers at once — shipping
full-resolution originals to every browser tab would be needless bandwidth and slow paint, so the
expensive resize work happens once, off the critical upload path, in a worker that can be scaled
independently of `api`. `ResizeImage` is idempotent (skips if `ImageResized` is already `true`)
and, on success, chains straight into `append-image-tags` — which is why the AI description in
§4 never fires until the thumbnail exists.

Open the console and browse both buckets live:
[http://minio.mytravels.local:8080](http://minio.mytravels.local:8080)
(`user123` / `password123`) — you'll see one object land in `uploaded-images` immediately and a
second, smaller one appear in `resized-images` a few seconds later.

**A live bug worth pointing at:** `GET /api/pointofinterest/{id}?resizedImage=` takes that flag on
the route but the controller never passes it down to the service — both values return the exact
same image. It's a real, catalogued finding (`SPEC.md` F-2), not a demo artifact.

In [ ]:
%%bash
API="http://api.mytravels.local:8080"
ID=$(curl -s "$API/api/pointofinterest?rows=200" | python3 -c "
import json, sys
pois = json.load(sys.stdin)
print(pois[-1]['id'] if pois else '')
")
if [ -z "$ID" ]; then
  echo "No points of interest yet — run the upload cell above first."
  exit 0
fi

echo "=== Same endpoint, resizedImage=true vs false — id=$ID ==="
LEN_TRUE=$(curl -s "$API/api/pointofinterest/$ID?resizedImage=true" | wc -c)
LEN_FALSE=$(curl -s "$API/api/pointofinterest/$ID?resizedImage=false" | wc -c)
echo "  resizedImage=true  -> $LEN_TRUE bytes (base64)"
echo "  resizedImage=false -> $LEN_FALSE bytes (base64)"
if [ "$LEN_TRUE" = "$LEN_FALSE" ]; then
  echo "  identical — the flag is accepted but never used (SPEC.md finding F-2)"
fi

---

## 4 — Integrating an LLM: Descriptions, Tags, and MCP

**The app calling an LLM.** Once `ResizeImage` finishes, it chains `append-image-tags`.
`AppendImageTags` checks the `enable-image-description` Flagsmith flag
([http://flagsmith.mytravels.local:8080](http://flagsmith.mytravels.local:8080), default `true`,
fails open if Flagsmith is unreachable) and, if it's on, sends the **original** image to Claude —
`AnthropicImageDescriptionService`, structured output pinned to `{ description, tags[] }`, model
`claude-haiku-4-5` by default — and persists both fields. The Anthropic key lives **only** on
`messaging`; `api` and `mcp` never see it, because they never make the call. Turning the flag off
is a clean way to run this whole stack with no LLM key configured at all: uploads, addresses and
thumbnails still work, only the description/tags step is skipped — and `index-solr` still fires
either way, so the POI stays searchable.

**The LLM calling the app.** `mcp` exposes the *same* domain services as two read-only MCP tools
over streamable HTTP — no REST surface, no Swagger, its own ingress host — so an MCP-aware LLM
client (Claude Desktop, an agent) can query this data directly:

| Tool | Arguments | Backed by |
|---|---|---|
| `search_pointofinterest` | `term` (not `query`) | the same SOLR index as `/search` |
| `search_place` | `query`, `limit` | the same maps service as `GET /api/place` |

It's the same asymmetry as the description feature, mirrored: one side of the app talks *to* an
LLM, the other side *is* a tool an LLM talks to — and both are anonymous, unauthenticated, and
entirely separate code paths that happen to share the same domain services underneath.

In [ ]:
%%bash
API="http://api.mytravels.local:8080"
ID=$(curl -s "$API/api/pointofinterest?rows=200" | python3 -c "
import json, sys
pois = json.load(sys.stdin)
print(pois[-1]['id'] if pois else '')
")
if [ -z "$ID" ]; then
  echo "No points of interest yet — run the upload cell above first."
  exit 0
fi

echo "=== Polling id=$ID for description/tags (calls Claude — can take up to a minute) ==="
for i in $(seq 1 12); do
  RESULT=$(curl -s "$API/api/pointofinterest?rows=200" | python3 -c "
import json, sys
pois = json.load(sys.stdin)
poi = next((p for p in pois if p['id'] == $ID), None)
if poi and poi.get('description'):
    print(f\"description={poi['description']!r}\")
    print('tags=' + ', '.join(t['name'] for t in poi.get('tags') or []))
"
  )
  if [ -n "$RESULT" ]; then
    echo "$RESULT"
    exit 0
  fi
  echo "  attempt $i: nothing yet..."
  sleep 5
done

echo "No description after 60s — this is normal if enable-image-description is off, or if"
echo "ANTHROPIC_API_KEY is a placeholder. Diagnosing inline:"
kubectl logs -n mytravels-default -l app=messaging --tail=60 | grep -i -e AppendImageTags -e anthropic || \
  kubectl logs -n mytravels-default -l app=messaging --tail=60

In [ ]:
%%bash
echo "=== mcp health (its only plain-HTTP route — everything else is MCP-over-HTTP) ==="
curl -s http://mcp.mytravels.local:8080/health
echo ""

---

## 5 — Enabling Search: SOLR

Since `api:v1.0.12`, **every POI read goes through SOLR** — the map's own list
(`GET /api/pointofinterest`), free-text search (`GET /api/pointofinterest/search`), and the MCP
tool above. PostgreSQL's `ILIKE` search is gone; SOLR is a *derived* store, rebuildable from
Postgres at any time via `POST /api/pointofinterest/reindex`.

Two things make this async pipeline converge instead of racing itself:

- **Documents are keyed on `PointOfInterestKey`, not the row id** — replacing a photo re-uses the
  key, so a rebuild and incremental indexing always agree on one document per key.
- **`index-solr` fires three times per upload** (creation, then the tail of address resolution,
  then the tail of tagging) because those fields all arrive on different timelines and SOLR
  upserts by key — repeat indexing is free.

Relevance boosts (`formatted_address^5 tags^3 description^1`) are query-time, so retuning ranking
needs no reindex. `rows` defaults to 100 and is silently truncated past that on both the list and
`/search` — the accepted cost of one capped query instead of a paging loop.

In [ ]:
%%bash
SOLR="http://solr.mytravels.local:8080"
API="http://api.mytravels.local:8080"

INDEXED=$(curl -s "$SOLR/solr/mytravels-pois/select?q=*:*&rows=0" | python3 -c "
import json, sys
print(json.load(sys.stdin)['response']['numFound'])
" 2>/dev/null || echo "?")
KEYS=$(curl -s "$API/api/pointofinterest?rows=200" | python3 -c "
import json, sys
print(len({p['pointOfInterestKey'] for p in json.load(sys.stdin)}))
")
echo "=== SOLR documents: $INDEXED   |   distinct keys via the API: $KEYS ==="

echo ""
echo "=== Free-text relevance search (address ^5, tags ^3, description ^1) ==="
for TERM in beach city mountain; do
  N=$(curl -s "$API/api/pointofinterest/search?term=$TERM&rows=50" | python3 -c "
import json, sys
print(len(json.load(sys.stdin)))
" 2>/dev/null || echo 0)
  printf "  term=%-10s %s result(s)\n" "$TERM" "$N"
done

---

## 6 — How Do You Know When a Message Fails? Traceability

Every message above carried the same `CorrelationId`. `MessagePublisher` writes a `Published` row
before every publish; `MessageSubscriberBase<T>` writes `ConsumeSucceeded`, `Retried`, or `Failed`
after every consume attempt — and that audit logging is wrapped in its own try/catch, so a logging
failure can never block the pipeline it's describing (the inverse also holds: a gap in the
timeline is not proof a step didn't run).

**The retry policy lives in application code, not the broker** — there is no dead-letter exchange,
no TTL, no max-length on any queue. On failure, `MessageSubscriberBase<T>` republishes to the
*same* exchange with an incremented `x-retry-count` header, immediately, no backoff. On the fourth
attempt it publishes a `FailedMessage` to the paired `<exchange>-failed` exchange and acks the
original — but **nothing is bound to those `-failed` exchanges**, so the message itself is
discarded on arrival. The `Failed` audit row is the only durable trace that it ever happened.

Read side: [http://web.mytravels.local:8080/traceability](http://web.mytravels.local:8080/traceability),
or `GET /api/traceability` / `GET /api/traceability/{id}` directly — both gated behind the
`enable-message-tracing` flag (writes are unaffected; only the read side 404s when it's off).

Pull the full timeline for the upload from §1:

In [ ]:
%%bash
API="http://api.mytravels.local:8080"
CORR=$(cat /tmp/inside_scoop_correlation_id.txt 2>/dev/null)
if [ -z "$CORR" ]; then
  echo "No correlation id saved — run the '=== Newest point of interest ===' cell in §1 first."
  exit 0
fi

echo "=== Event timeline for correlationId=$CORR ==="
curl -s "$API/api/traceability/$CORR" | python3 -c "
import json, sys
events = json.load(sys.stdin)
if not events:
    print('  (no audit rows yet — the pipeline may still be in flight)')
for e in events:
    marker = '  FAILED  ' if e['eventType'] == 'Failed' else ('  retried ' if e['eventType'] == 'Retried' else '          ')
    print(f\"{marker}{e['createdAt']}  {e['exchangeName']:<28} {e['eventType']:<16} {e.get('errorMessage') or ''}\")
"
echo ""
echo "That's one photo upload, traced end-to-end across every service that touched it."

### Optional live demo — force a failure

This is destructive to `messaging`'s config until reverted; skip it if you'd rather not touch the
live stack mid-talk.

1. Point `AnthropicApiKey` at garbage: `kubectl set env deployment/messaging -n mytravels-default AnthropicApiKey=broken`
   — this is drift from what's in git, and `selfHeal` is `false` (per `runbook.ipynb` Step 13),
   so Argo CD will *not* undo it for you; it'll just show the Application as `OutOfSync`.
2. Upload a new photo. `AppendImageTags` now gets a `503` from `AnthropicImageDescriptionService`
   on every attempt — 3 immediate retries, then a `Failed` audit row, visible in
   `/traceability` with `hasFailure: true`. Addresses and thumbnails still work; only tagging
   fails, because the three enrichment steps are fully independent messages.
3. Revert: `kubectl apply -f manifests/messaging/deployment.yaml` restores the tracked
   `secretKeyRef` (a plain `-` removal would just delete the env var entirely, not restore it).

---

## 7 — What Fails First When Traffic Increases?

RabbitMQ is the shock absorber — an upload burst just makes queues deeper, not the API slower.
But depth isn't free capacity; three things bound how fast that depth drains:

- **Each subscriber class serializes itself** behind a `static SemaphoreSlim(1,1)`, so
  `AppendFormattedAddress`, `ResizeImage`, and `AppendImageTags` each process **one message at a
  time per process**, regardless of RabbitMQ's `prefetchCount: 10`. The three pipelines run in
  parallel *with each other*; each one alone is strictly serial.
- **`AppendImageTags` is the tightest of the three** — it's not just serialized, it's also waiting
  on a metered third-party call (Claude) every time, so its queue is the one that visibly backs up
  first under load.
- **`messaging` runs 2 static replicas, with no autoscaler anywhere in these manifests** — so
  under sustained load, the fix is a manual `kubectl scale`, not a HorizontalPodAutoscaler
  reacting to queue depth.

One more compounding factor on the producer side: `MessagePublisher` opens a brand-new AMQP
connection and channel **per publish call** — no pooling — so the cost of publishing itself grows
with traffic too, on top of the consumer-side serialization.

Watch queue depth live: [http://rabbitmq.mytravels.local:8080](http://rabbitmq.mytravels.local:8080)
(`user123` / `password123`), or pull it here:

In [ ]:
%%bash
# Point PHOTOS_DIR at a folder of geotagged photos.
# The cell skips itself if the folder does not exist, so it is safe to run top-to-bottom.
PHOTOS_DIR="${PHOTOS_DIR:-$HOME/Personal/photos}"
API_BASE="http://api.mytravels.local:8080"

if [ ! -d "$PHOTOS_DIR" ]; then
  echo "SKIPPED — not a directory: $PHOTOS_DIR"
  echo "Set PHOTOS_DIR above to a folder of geotagged photos and re-run this cell to seed data."
  exit 0
fi

if ! curl -s -o /dev/null -m 5 "$API_BASE/api/pointofinterest"; then
  echo "API not reachable at $API_BASE — diagnosing inline:"
  echo "--- Application (a failed sync looks like a down API) ---"
  kubectl get application mytravels -n argocd
  kubectl get application mytravels -n argocd \
    -o jsonpath='{.status.operationState.phase}: {.status.operationState.message}{"\n"}'
  echo "--- pods (app=api) ---"
  kubectl get pods -n mytravels-default -l app=api -o wide
  echo "--- api logs (last 30) ---"
  kubectl logs -n mytravels-default -l app=api --tail=30
  exit 1
fi

echo "=== Uploading photos from: $PHOTOS_DIR ==="
../.claude/scripts/upload-photos.sh "$PHOTOS_DIR" "$API_BASE"

In [ ]:
%%bash
RMQ="http://rabbitmq.mytravels.local:8080"
USER=$(grep -E '^RABBITMQ_DEFAULT_USER=' .env | cut -d= -f2-)
PASS=$(grep -E '^RABBITMQ_DEFAULT_PASS=' .env | cut -d= -f2-)

echo "=== Queue depth per pipeline (work queues, named after their exchange) ==="
curl -s -u "$USER:$PASS" "$RMQ/api/queues" | python3 -c "
import json, sys
queues = json.load(sys.stdin)
for q in sorted(queues, key=lambda q: q['name']):
    print(f\"  {q['name']:<28} ready={q.get('messages_ready', 0):<5} unacked={q.get('messages_unacknowledged', 0):<5} consumers={q.get('consumers', 0)}\")
" 2>/dev/null || echo "Could not reach the management API — check .env has RABBITMQ_DEFAULT_USER/PASS."

echo ""
echo "=== messaging replica count (static — no HPA in these manifests) ==="
kubectl get deployment messaging -n mytravels-default -o jsonpath='replicas: {.spec.replicas}{"\n"}'

---

## 8 — Monitor Your Resources: Prometheus & Grafana

Every application service exports OTLP traces and metrics; `postgres-exporter` and `cAdvisor` add
database and container-level metrics; Prometheus scrapes all of it; Grafana renders the
**"MyTravels Overview"** dashboard — which happens to be exactly the signals from §7's bottleneck
story, in one place:

- API request rate
- API error rate (5xx)
- **RabbitMQ queue depth** — the same numbers as the cell above, over time
- Postgres active connections
- Container CPU usage

[http://grafana.mytravels.local:8080](http://grafana.mytravels.local:8080) ·
[http://prometheus.mytravels.local:8080](http://prometheus.mytravels.local:8080)
(`user123` / `password123`)

In [ ]:
%%bash
echo "=== Prometheus scrape targets ==="
curl -s http://prometheus.mytravels.local:8080/api/v1/targets \
  | python3 -c "
import json, sys
data = json.load(sys.stdin)
for t in data['data']['activeTargets']:
    print(f\"  {t['labels'].get('job'):<20} {t['health']:<8} {t.get('lastError', '')}\")
"

---

## 9 — Monitoring the Cluster Itself: Argo CD

Everything above is only running because Argo CD continuously reconciles the cluster against
`manifests/` on git — sync waves order the dependencies (namespace → secrets → postgres →
migrations → everything else), and self-heal reverts manual drift on its own. That mechanism is
the subject of the *next* part of this talk (Kubernetes); here it's worth showing once, live,
because it's the same "who's watching this system" question as §6 and §8, aimed at the platform
instead of the app.

[http://argocd.mytravels.local:8080](http://argocd.mytravels.local:8080)
(`user123` / `password123`) — full drift/self-heal walkthrough:
[runbook.ipynb, Step 13](runbook.ipynb).

In [ ]:
%%bash
echo "=== Application health/sync, as Argo sees it right now ==="
kubectl get application mytravels -n argocd \
  -o jsonpath='sync={.status.sync.status}  health={.status.health.status}{"\n"}'
echo ""
echo "=== Resources it manages ==="
kubectl get application mytravels -n argocd \
  -o jsonpath='{range .status.resources[*]}{.kind}{"/"}{.name}{" "}{.status}{"\n"}{end}'

---

## 10 — Integrations to Other Systems

Every external dependency in this app sits behind its own service boundary with its own failure
mode — nothing is a straight-line call from the browser to a third party:

| Integration | Called from | Failure mode |
|---|---|---|
| Google Maps Geocoding (+ OSM fallback) | `messaging` | Polly retries, then a dead-lettered message — addresses/thumbnails elsewhere are unaffected |
| Anthropic Claude | `messaging` only | fails open behind a Flagsmith flag; a bad key dead-letters `append-image-tags` alone |
| MCP clients (Claude Desktop, agents) | inbound, to `mcp` | read-only, anonymous, isolated from `api`'s write surface entirely |
| Flagsmith | `api`, `messaging`, `web` | fails open — an outage means every gated feature defaults to on, not a 500 |
| OTLP collector | `api`, `mcp`, `messaging`, browser RUM | export failures are logged and swallowed, never fatal |

The pattern repeats: **one external dependency going down degrades one feature, not the app.**

---

## Wrap-Up — What This Buys You, What It Costs

**What you get** from pulling one action apart into five independent services:

- Each piece scales and deploys on its own — `messaging` replicas without touching `api`.
- A slow or failing third party (Claude, a geocoder) degrades one feature, not the whole upload.
- Providers swap by config, not code — Google↔OSM, feature flags for the AI path, model name via
  a ConfigMap.
- A single action is fully traceable end-to-end across process boundaries via one correlation id,
  even though no two of these services share a transaction.

**What it costs:**

- Eventual consistency, everywhere — a POI exists in Postgres before it's on the map, before it
  has an address, before it has a description. The frontend has to poll around that gap.
- No broker-level safety net — no DLX, no TTL; the retry/dead-letter policy is application code,
  and "did it work?" only has an answer because someone built §6.
- More moving parts to actually operate — ten exchanges, five services, a bottleneck (§7) that
  only shows up under load and only if someone's watching (§8).
- All of this needs *something* holding it together reliably in production — which is exactly
  where Kubernetes comes in next.